# Optiver — Next Steps: Trade Features & Scaling Past 4 Stocks

This is a **new** notebook that continues from `explore_data.ipynb` (which is left
untouched). It tackles the two "next steps" named at the end of the README:

1. **Trade-data features** — so far we only used the *order book* (resting buy/sell
   intentions). Here we add features from `trade_train.parquet`, the **actual
   executed trades**.
2. **Scaling past 4 stocks** — the original work used stocks `[0, 1, 2, 3]`. Here we
   use the first **20** stocks to check the pipeline still behaves at larger scale.

**How to run:** *Restart & Run All*. It reads ~40 parquet files (book + trade for 20
stocks), so the big build cell takes roughly **30 seconds**.
We deliberately do NOT load all 112 stocks by default (see the last cell for how).

## 1. Setup

We read the list of stock IDs that actually exist on disk (they are **not** a tidy
`0..111` there are gaps), then take the first 20.

In [1]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

# Stock folders look like "stock_id=0"; the numbers are not contiguous, so list them.
available_stock_ids = sorted(
    int(name.split("=")[1])
    for name in os.listdir("data/book_train.parquet")
    if name.startswith("stock_id=")
)
print("total stocks available:", len(available_stock_ids))

SUBSET_STOCK_IDS = available_stock_ids[:20]   # scale past the original 4 -> first 20
print("using:", SUBSET_STOCK_IDS)

total stocks available: 112
using: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20]


## 2. What is trade data?

The **order book** we used before is a list of *resting* orders (prices people are
*willing* to trade at, but have not necessarily acted on). **Trade data** is different:
each row is a trade that **actually happened**.

`trade_train.parquet` columns:

| column | meaning |
|---|---|
| `time_id` | which 10-minute window |
| `seconds_in_bucket` | when in the window the trade happened |
| `price` | the price the trade executed at |
| `size` | how many shares changed hands |
| `order_count` | how many individual orders made up this trade |

Let's look at one stock.

In [2]:
trades_stock_0 = pd.read_parquet("data/trade_train.parquet/stock_id=0")
print(trades_stock_0.shape)
trades_stock_0.head()

(123443, 5)


,time_id,seconds_in_bucket,price,size,order_count
0,5,21,1.002301,326,12
1,5,46,1.002778,128,4
2,5,50,1.002818,55,1
3,5,57,1.003155,121,5
4,5,68,1.003646,4,1


## 3. The order-book features (recap)

This is the **same** function from `explore_data.ipynb` (M3/M4) one row of
order-book features per `(stock_id, time_id)` window. Nothing new here, we just need
it available in this notebook too.

In [3]:
def compute_book_features(stock_id):
    """One row of ORDER-BOOK features per (stock_id, time_id) window (same as M3/M4)."""
    book = pd.read_parquet(f"data/book_train.parquet/stock_id={stock_id}")

    # Weighted average price: fair mid-price, each side weighted by the OTHER side's size.
    book["wap"] = (
        (book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
        / (book["bid_size1"] + book["ask_size1"])
    )
    # Log return between consecutive snapshots, WITHIN each window only.
    book["log_return"] = np.log(book["wap"] / book.groupby("time_id")["wap"].shift(1))
    book["spread"] = book["ask_price1"] - book["bid_price1"]
    book["size_imbalance"] = book["bid_size1"] / (book["bid_size1"] + book["ask_size1"])

    # Volatility over just the last 300s (closest in time to the window we predict).
    last_300 = book[book["seconds_in_bucket"] >= 300]
    rv_last_300 = last_300.groupby("time_id")["log_return"].apply(lambda x: np.sqrt((x ** 2).sum()))

    grouped = book.groupby("time_id")
    features = pd.DataFrame({
        "realized_vol": grouped["log_return"].apply(lambda x: np.sqrt((x ** 2).sum())),
        "realized_vol_last_300": rv_last_300,
        "spread_mean": grouped["spread"].mean(),
        "size_imbalance_mean": grouped["size_imbalance"].mean(),
    }).reset_index()

    features["realized_vol_last_300"] = features["realized_vol_last_300"].fillna(features["realized_vol"])
    features["stock_id"] = stock_id
    return features

## 4. The NEW trade features

Per `(stock_id, time_id)` window we compute:

- **`trade_count`** — how many trades happened (activity).
- **`total_volume`** — total shares traded (how much actually changed hands).
- **`mean_trade_size`** — average trade size.
- **`trade_order_count`** — total orders behind those trades.
- **`trade_vol`** — realized volatility computed from the **trade prices** themselves.
  This is a *second, independent* volatility signal: the order-book `realized_vol`
  measures how the quoted mid-price moved; `trade_vol` measures how the prices people
  actually **traded at** moved. We expect this to be the strongest of the five.

In [4]:
def compute_trade_features(stock_id):
    """One row of TRADE features per (stock_id, time_id) window."""
    trades = pd.read_parquet(f"data/trade_train.parquet/stock_id={stock_id}")

    # Realized volatility from the prices trades ACTUALLY executed at.
    trades["log_return"] = np.log(trades["price"] / trades.groupby("time_id")["price"].shift(1))

    grouped = trades.groupby("time_id")
    features = pd.DataFrame({
        "trade_count": grouped.size(),                      # number of trades
        "total_volume": grouped["size"].sum(),              # total shares traded
        "mean_trade_size": grouped["size"].mean(),          # average trade size
        "trade_order_count": grouped["order_count"].sum(),  # orders behind the trades
        "trade_vol": grouped["log_return"].apply(lambda x: np.sqrt((x ** 2).sum())),
    }).reset_index()

    features["stock_id"] = stock_id
    return features

## 5. Build & combine the tables (this is the slow cell, ~30s)

We build both tables for all 20 stocks and join them. A **left join** keeps every
order-book window and attaches trade features where they exist.

Some windows have **no trades at all**. Those trade columns come out as `NaN`. A
window with no trades genuinely has 0 trades / 0 volume / 0 trade-volatility, so we
fill those with `0` (not the column mean, that would invent activity that did not
happen).

In [5]:
train = pd.read_csv("data/train.csv")

# Reads ~40 parquet files (book + trade for each of 20 stocks) -> ~30 seconds.
book_table = pd.concat([compute_book_features(s) for s in SUBSET_STOCK_IDS], ignore_index=True)
trade_table = pd.concat([compute_trade_features(s) for s in SUBSET_STOCK_IDS], ignore_index=True)
print("book table :", book_table.shape)
print("trade table:", trade_table.shape)

# Left join keeps every book window; trade features attach where available.
features = book_table.merge(trade_table, on=["stock_id", "time_id"], how="left")

trade_cols = ["trade_count", "total_volume", "mean_trade_size", "trade_order_count", "trade_vol"]
features[trade_cols] = features[trade_cols].fillna(0)   # no-trade windows -> 0

# Same derived book feature as before, then attach the true target.
features["imbalance_intensity"] = (features["size_imbalance_mean"] - 0.5).abs()
features = features.merge(train, on=["stock_id", "time_id"])
print("combined feature table:", features.shape)
features.head()

book table : (76599, 6)
trade table: (76598, 7)
combined feature table: (76599, 13)


,time_id,realized_vol,realized_vol_last_300,spread_mean,size_imbalance_mean,stock_id,trade_count,total_volume,mean_trade_size,trade_order_count,trade_vol,imbalance_intensity,target
0,5,0.004499,0.002953,0.000855,0.496821,0,40.0,3179.0,79.475000,110.0,0.002006,0.003179,0.004136
1,11,0.001204,0.000981,0.000394,0.583582,0,30.0,1289.0,42.966667,57.0,0.000901,0.083582,0.001445
2,16,0.002369,0.001295,0.000725,0.447253,0,25.0,2161.0,86.440000,68.0,0.001961,0.052747,0.002168
3,31,0.002574,0.001776,0.000859,0.528189,0,15.0,1962.0,130.800000,59.0,0.001561,0.028189,0.002195
4,62,0.001894,0.001520,0.000397,0.544335,0,22.0,1791.0,81.409091,89.0,0.000871,0.044335,0.001747


## 6. Do the trade features move with the target?

A quick correlation check before modelling. We expect `trade_vol` to lead.

In [6]:
print(features[trade_cols + ["target"]].corr()["target"].drop("target").sort_values(ascending=False))

trade_vol            0.811461
trade_order_count    0.226150
total_volume         0.154305
trade_count          0.098333
mean_trade_size      0.066320
Name: target, dtype: float64


## 7. Do they actually improve the model?

We score three things under the **same honest cross-validation** used before
(`GroupKFold` split by `time_id`, so no market moment leaks across folds; tuned
LightGBM; `1/target²` weighting to optimise the RMSPE percentage error):

1. the naive baseline,
2. a model with **book features only**,
3. a model with **book + trade features**.

If the trade features add real signal, (3) beats (2).

In [7]:
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

book_cols = ["realized_vol", "realized_vol_last_300", "spread_mean",
             "size_imbalance_mean", "imbalance_intensity"]

def cross_validated_rmspe(feature_columns):
    """Pooled out-of-fold RMSPE with the tuned model + GroupKFold-by-time_id."""
    X = features[feature_columns]
    y = features["target"]
    groups = features["time_id"]
    gkf = GroupKFold(n_splits=5)
    oof_pred = np.zeros(len(features))
    for train_idx, val_idx in gkf.split(X, y, groups):
        model = lgb.LGBMRegressor(
            n_estimators=126, learning_rate=0.0437, num_leaves=15,   # tuned in explore_data.ipynb
            random_state=42, verbose=-1, n_jobs=1,
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx],
                  sample_weight=1.0 / np.square(y.iloc[train_idx]))
        oof_pred[val_idx] = model.predict(X.iloc[val_idx])
    return rmspe(y.values, oof_pred)

naive_rmspe = rmspe(features["target"].values, features["realized_vol"].values)
book_only_rmspe = cross_validated_rmspe(book_cols)
book_trade_rmspe = cross_validated_rmspe(book_cols + trade_cols)

print(f"naive baseline      : {naive_rmspe:.5f}")
print(f"book-only model     : {book_only_rmspe:.5f}")
print(f"book + trade model  : {book_trade_rmspe:.5f}")
print(f"trade features help : {(1 - book_trade_rmspe / book_only_rmspe) * 100:.1f}% lower error")

naive baseline      : 0.33763
book-only model     : 0.24966
book + trade model  : 0.24363
trade features help : 2.4% lower error


## 8. Which features does the model lean on?

LightGBM can report a **feature importance** (roughly, how much each feature improved
its predictions). This tells us *which* of the new trade features earned their place.

In [8]:
X = features[book_cols + trade_cols]
y = features["target"]
model = lgb.LGBMRegressor(n_estimators=126, learning_rate=0.0437, num_leaves=15,
                          random_state=42, verbose=-1, n_jobs=1)
model.fit(X, y, sample_weight=1.0 / np.square(y))

importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance)

realized_vol             339
realized_vol_last_300    285
mean_trade_size          259
spread_mean              246
trade_vol                237
trade_order_count        182
trade_count              123
total_volume              39
size_imbalance_mean       37
imbalance_intensity       17
dtype: int32


## 9. What we found

Running the cells above (deterministic, so you should see these numbers):

- **naive baseline ≈ 0.338**, **book-only ≈ 0.250**, **book + trade ≈ 0.244**;
  the trade features cut error a further **~2.4%**.
- The value comes almost entirely from **`trade_vol`** (correlation ~0.81 with the
  target); volatility of the *actual traded prices* is a genuinely new signal beyond
  the order-book WAP volatility. `trade_order_count` and `total_volume` add a little;
  `trade_count` and `mean_trade_size` add almost nothing.
- **Scaling held up:** at 20 stocks the naive baseline (~0.338) and book-only model
  (~0.250) match what we saw on 4 stocks, so nothing broke or leaked at larger scale.

**Takeaway:** trade features help, but modestly. The same lesson as the
tuning step. The order-book realized volatility remains the dominant signal; trades
refine it rather than replace it.

### Going all the way to 112 stocks
Change the subset in the setup cell to `SUBSET_STOCK_IDS = available_stock_ids`
(all 112). Expect the big build cell to take a few minutes instead of ~30 seconds.
Everything else runs unchanged.